In [36]:
import pandas as pd

ratings = pd.read_csv(
    "ml-1m/ratings.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["userId", "movieId", "rating", "timestamp"],
    header=None
)

movies = pd.read_csv(
    "ml-1m/movies.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["movieId", "title", "genres"],
    header=None
)

In [37]:
movies["content"] = movies["title"] + " " + movies["genres"].str.replace("|", " ")
movies.head()

,movieId,title,genres,content
0,1,Toy Story (1995),Animation|Children's|Comedy,Toy Story (1995) Animation Children's Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy,Jumanji (1995) Adventure Children's Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men (1995) Comedy Romance
3,4,Waiting to Exhale (1995),Comedy|Drama,Waiting to Exhale (1995) Comedy Drama
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II (1995) Comedy


In [38]:
ratings

,userId,movieId,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291
...,...,...,...,...
1000204,6040,1091,1,956716541
1000205,6040,1094,5,956704887
1000206,6040,562,5,956704746
1000207,6040,1096,4,956715648


In [39]:
data = ratings.merge(movies, on="movieId")
data.head(5)

,userId,movieId,rating,timestamp,title,genres,content
0,1,1193,5,978300760,One Flew Over the Cuckoo's Nest (1975),Drama,One Flew Over the Cuckoo's Nest (1975) Drama
1,1,661,3,978302109,James and the Giant Peach (1996),Animation|Children's|Musical,James and the Giant Peach (1996) Animation Chi...
2,1,914,3,978301968,My Fair Lady (1964),Musical|Romance,My Fair Lady (1964) Musical Romance
3,1,3408,4,978300275,Erin Brockovich (2000),Drama,Erin Brockovich (2000) Drama
4,1,2355,5,978824291,"Bug's Life, A (1998)",Animation|Children's|Comedy,"Bug's Life, A (1998) Animation Children's Comedy"


Build the User–Movie Rating Matrix

In [40]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)
print(f"Train shape: {train_data.shape}, Test shape: {test_data.shape}")

Train shape: (800167, 4), Test shape: (200042, 4)


In [41]:
ratings_matrix = train_data.pivot(index="userId", columns="movieId", values="rating")
ratings_matrix = ratings_matrix.fillna(0)

SVD requires a numeric matrix.

In [42]:
import numpy as np

R = ratings_matrix.to_numpy()
print(R.shape)

(6040, 3683)


normalize rating using mean

In [43]:
user_mean = np.mean(R, axis = 1)
user_mean = user_mean.reshape(-1,1)
R_centered = R - user_mean

In [44]:
print(np.isnan(R_centered).sum())

0


Compute SVD 

In [45]:
U, sigma, Vt = np.linalg.svd(R_centered, full_matrices=False)

In [46]:
k = 50

U_k = U[:,:k]
sigma_k = np.diag(sigma[:k])
Vt_k = Vt[:k,:]

Reconstruct Predicted Rating Matrix

In [47]:
R_pred = U_k @ sigma_k @ Vt_k
R_pred = R_pred + user_mean


In [48]:
pred_df = pd.DataFrame(R_pred, index=ratings_matrix.index, columns=ratings_matrix.columns)


Recommend Movies for a User

In [49]:
user_id = 1
rated_movies = train_data[train_data["userId"] == user_id]["movieId"].tolist()
user_predictions = pred_df.loc[user_id]
user_predictions = user_predictions.drop(rated_movies)
top_movies = user_predictions.sort_values(ascending=False).head(20)

recommend_movies = movies[movies["movieId"].isin(top_movies.index)]["title"]

print(recommend_movies)

33                     Babe (1995)
352            Forrest Gump (1994)
360          Lion King, The (1994)
592               Pinocchio (1940)
887     Singin' in the Rain (1952)
1019    Alice in Wonderland (1951)
1022    Sound of Music, The (1965)
1262               Fantasia (1940)
1656      Good Will Hunting (1997)
1878        West Side Story (1961)
1949                  Bambi (1942)
2009       Jungle Book, The (1967)
2011     Lady and the Tramp (1955)
2012    Little Mermaid, The (1989)
2016         101 Dalmatians (1961)
2018              Peter Pan (1953)
2027        Sleeping Beauty (1959)
2647           Ghostbusters (1984)
2692        Iron Giant, The (1999)
3682            Chicken Run (2000)
Name: title, dtype: object


Evaluation process

In [50]:
users = test_data["userId"].unique()

recalls = []
precisions = []

for user in users:
    if user not in pred_df.index:
        continue

    user_test = test_data[test_data["userId"] == user]
    user_train = train_data[train_data["userId"] == user]

    test_movies = user_test[user_test["rating"] >= 4]["movieId"].tolist()
    
    if len(test_movies) == 0:
        continue

    train_movies = user_train["movieId"].tolist()

    user_predictions = pred_df.loc[user]
    user_predictions = user_predictions.drop(labels=train_movies, errors="ignore")

    top10 = user_predictions.sort_values(ascending=False).head(10)
    recommended = top10.index.tolist()

    hits = len(set(recommended) & set(test_movies))

    recall = hits / len(test_movies)
    precision = hits / 10

    recalls.append(recall)
    precisions.append(precision)


In [51]:
avg_recall = np.mean(recalls)
avg_precision = np.mean(precisions)

print("Average Recall@10:", avg_recall)
print("Average Precision@10:", avg_precision)

Average Recall@10: 0.19694656382519865
Average Precision@10: 0.2765765765765766


### score without spliting the data.
---
Average Recall@10: 0.30454275505123

Average Precision@10: 0.39395094464700037


### score after spliting the data. 
---
Average Recall@10: 0.19694656382519865

Average Precision@10: 0.2765765765765766